In [0]:
%run ./config

✅ Config loaded


In [0]:


# ============================================
# CONFIGURATION - CHANGE FOR TEST/PROD
# ============================================

# Set to True for testing, False for production
TEST_MODE = True  # ← Change to False for production

STORAGE_ACCOUNT = "funddatalakeshantanu"

if TEST_MODE:
    CATALOG_NAME = "workspace"
    SILVER_SCHEMA = "default"
    GOLD_SCHEMA = "default"
    print("🧪 TEST MODE ENABLED")
else:
    CATALOG_NAME = "stock_exchange_api_catalog"
    SILVER_SCHEMA = "silver"
    GOLD_SCHEMA = "gold"
    print("🏭 PRODUCTION MODE ENABLED")

print(f"✅ Using catalog: {CATALOG_NAME}")
print(f"✅ Silver schema: {SILVER_SCHEMA}")
print(f"✅ Gold schema: {GOLD_SCHEMA}")

# Note: ADLS configuration removed - not needed on AWS Serverless compute
# Data will be read/written to Unity Catalog tables

# ============================================
# READ FROM SILVER
# ============================================

from pyspark.sql.functions import avg, min, max, stddev, count, sum, col

meta_df = spark.table(f"{CATALOG_NAME}.{SILVER_SCHEMA}.stock_meta")
quote_df = spark.table(f"{CATALOG_NAME}.{SILVER_SCHEMA}.stock_quotes")

print("✅ Data loaded from Silver")
print(f"Metadata: {meta_df.count()} records")
print(f"Quotes: {quote_df.count()} records")

# ============================================
# CALCULATE KPIs
# ============================================

daily_summary = quote_df.groupBy("symbol").agg(
    avg("close").alias("avg_price"),
    min("close").alias("min_price"),
    max("close").alias("max_price"),
    stddev("close").alias("volatility"),
    count("close").alias("trading_minutes"),
    sum("volume").alias("total_volume")
)

avg_volume = quote_df.groupBy("symbol").agg(avg("volume").alias("avg_volume"))
daily_summary = daily_summary.join(avg_volume, on="symbol", how="left")
daily_summary = daily_summary.withColumn("turnover_ratio", col("total_volume") / col("avg_volume"))

gold_df = daily_summary.join(
    meta_df.select("symbol", "current_price", "currency", "exchange"), 
    on="symbol", 
    how="left"
)

print("✅ Gold transformation complete")
display(gold_df)

# ============================================
# SAVE TO GOLD
# ============================================

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{GOLD_SCHEMA}")

gold_df.write.mode("overwrite").saveAsTable(f"{CATALOG_NAME}.{GOLD_SCHEMA}.gold_summary")

print(f"✅ Gold table saved to {CATALOG_NAME}.{GOLD_SCHEMA}")

# ============================================
# VERIFY
# ============================================

print("\n📊 Gold Table:")
display(spark.sql(f"SELECT * FROM {CATALOG_NAME}.{GOLD_SCHEMA}.gold_summary"))

print("\n📊 Quick Insights:")
gold_verify = spark.sql(f"SELECT * FROM {CATALOG_NAME}.{GOLD_SCHEMA}.gold_summary")
if gold_verify.count() > 0:
    print(f"  📈 Total stocks: {gold_verify.count()}")
    print(f"  🏆 Top performer: {gold_verify.orderBy(col('avg_price').desc()).select('symbol').first()[0]}")
    print(f"  📊 Most volatile: {gold_verify.orderBy(col('volatility').desc()).select('symbol').first()[0]}")
    print(f"  🔄 Highest turnover: {gold_verify.orderBy(col('turnover_ratio').desc()).select('symbol').first()[0]}")

print("\n✅ Gold layer complete!")

🧪 TEST MODE ENABLED
✅ Using catalog: workspace
✅ Silver schema: default
✅ Gold schema: default
✅ Data loaded from Silver
Metadata: 5 records
Quotes: 1955 records
✅ Gold transformation complete


symbol,avg_price,min_price,max_price,volatility,trading_minutes,total_volume,avg_volume,turnover_ratio,current_price,currency,exchange
AAPL,302.5495463417619,300.0799865722656,309.7200012207031,1.7682716905870701,391,112910987,288774.9028132992,391.00000000000006,308.91,USD,NMS
MSFT,460.3456920418898,451.20001220703125,466.57000732421875,3.4310603970252576,391,48294724,123515.91815856777,391.0,464.72,USD,NMS
GOOG,351.9321452967651,340.5899963378906,358.7200012207031,4.127761752332774,391,25897896,66235.02813299233,391.0,356.65,USD,NMS
TSLA,309.01960563171855,302.17999267578125,315.0,2.4086804421298202,391,33071956,84583.00767263427,391.0,311.21,USD,NMS
NVDA,198.22652144566217,195.17999267578125,201.88499450683594,1.4728625789161458,391,109007188,278790.7621483376,391.0,200.75,USD,NMS


✅ Gold table saved to workspace.default

📊 Gold Table:


symbol,avg_price,min_price,max_price,volatility,trading_minutes,total_volume,avg_volume,turnover_ratio,current_price,currency,exchange
AAPL,302.5495463417619,300.0799865722656,309.7200012207031,1.7682716905870701,391,112910987,288774.9028132992,391.00000000000006,308.91,USD,NMS
MSFT,460.3456920418898,451.20001220703125,466.57000732421875,3.4310603970252576,391,48294724,123515.91815856777,391.0,464.72,USD,NMS
GOOG,351.9321452967651,340.5899963378906,358.7200012207031,4.127761752332774,391,25897896,66235.02813299233,391.0,356.65,USD,NMS
TSLA,309.01960563171855,302.17999267578125,315.0,2.4086804421298202,391,33071956,84583.00767263427,391.0,311.21,USD,NMS
NVDA,198.22652144566217,195.17999267578125,201.88499450683594,1.4728625789161458,391,109007188,278790.7621483376,391.0,200.75,USD,NMS



📊 Quick Insights:
  📈 Total stocks: 5
  🏆 Top performer: MSFT
  📊 Most volatile: GOOG
  🔄 Highest turnover: AAPL

✅ Gold layer complete!
